# Knee MRI Multi-Label Classification: Exploratory Data Analysis (EDA)

This notebook provides a complete exploratory workflow for the Knee MRI dataset:
- `data/sample/train.csv`: Study metadata, radiology reports, and 12 binary target pathology conditions.
- `data/sample/train_series.csv`: MRI series acquisition parameters (planes, fluid sensitivity flags).
- `data/sample/*.dcm`: Sample DICOM files.

In [ ]:
# ------------------------------------------------------------------------------
# Step 1: Environment Setup & Target Definitions
# ------------------------------------------------------------------------------
import os
import sys
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom

# The 12 multi-label binary targets
TARGET_COLUMNS = [
    "ACL",              # Anterior Cruciate Ligament tear / pathology
    "MCL",              # Medial Collateral Ligament tear / pathology
    "Medial Meniscus",  # Medial Meniscal tear / pathology
    "Lateral Meniscus", # Lateral Meniscal tear / pathology
    "Medial OA",        # Medial Compartment Osteoarthritis
    "Lateral OA",       # Lateral Compartment Osteoarthritis
    "PF OA",            # Patellofemoral Osteoarthritis
    "Effusion",         # Joint Effusion
    "Synovitis",        # Synovitis / Synovial Thickening
    "Baker's",          # Baker's Cyst
    "Contusion",        # Bone Marrow Edema / Contusion
    "Fracture",         # Bone Fracture
]

# Add root of repo to path to enable module imports
ROOT = Path(os.getcwd(), "..").resolve().as_posix()

if ROOT not in sys.path:
    sys.path.append(ROOT)

from config import Config
conf = Config()

DATA_DIR = Path(ROOT, conf.find("data.dir"))

In [ ]:
train_df = pd.read_csv(DATA_DIR / conf.find("data.train"))
series_df = pd.read_csv(DATA_DIR / conf.find("data.train_series"))

print(f"train.csv shape:        {train_df.shape}")
print(f"train_series.csv shape: {series_df.shape}")

## 2. Target Class & Imbalance Distribution
Extracting positive/negative prevalence, imbalance ratios, and co-occurrence across the labeled subset.

In [ ]:
# Filter labeled studies (N=58) from unlabeled studies (N=4,349)
labeled_df = train_df[train_df[TARGET_COLUMNS].notna().any(axis=1)].copy()
unlabeled_df = train_df[train_df[TARGET_COLUMNS].isna().all(axis=1)].copy()

print(f"Ground-truth labeled studies: {len(labeled_df)} ({len(labeled_df)/len(train_df)*100:.2f}%)")
print(f"Unlabeled studies (reports only): {len(unlabeled_df)} ({len(unlabeled_df)/len(train_df)*100:.2f}%)")

# Compute class statistics
stats = []
for col in TARGET_COLUMNS:
    pos = int(labeled_df[col].sum())
    neg = int((labeled_df[col] == 0).sum())
    stats.append({
        "Target Class": col,
        "Positive (1)": pos,
        "Negative (0)": neg,
        "Pos Prevalence (%)": f"{(pos / len(labeled_df)) * 100:.1f}%",
        "Imbalance (Neg:Pos)": f"{neg / pos:.2f}:1" if pos > 0 else "N/A"
    })

pd.DataFrame(stats)

In [ ]:
# Create barplot (horizontal bars)
import matplotlib.pyplot as plt

# Apply ggplot style
plt.style.use("ggplot")

# Reverse target order so the first target appears at the top
targets = TARGET_COLUMNS[::-1]
pos_counts = np.array([int(labeled_df[col].sum()) for col in targets])
neg_counts = np.array([int((labeled_df[col] == 0).sum()) for col in targets])
totals = pos_counts + neg_counts

fig, ax = plt.subplots(figsize=(10, 7))

# ggplot color palette: coral/red for Positive, blue for Negative
pos_color = "#E24A33"
neg_color = "#348ABD"

# Plot stacked horizontal bars
bars_pos = ax.barh(targets, pos_counts, color=pos_color, label="Positive (1)", height=0.65)
bars_neg = ax.barh(targets, neg_counts, left=pos_counts, color=neg_color, label="Negative (0)", height=0.65)

# Annotate actual number and percentage labels inside each bar segment
for i, (p, n, tot) in enumerate(zip(pos_counts, neg_counts, totals)):
    # Positive label
    ax.text(p / 2, i, f"{p} ({p/tot*100:.1f}%)", ha="center", va="center", color="white", fontweight="bold", fontsize=10)
    # Negative label
    ax.text(p + n / 2, i, f"{n} ({n/tot*100:.1f}%)", ha="center", va="center", color="white", fontweight="bold", fontsize=10)
    # Total count label at the end of the bar
    ax.text(tot + 0.8, i, f"Total: {tot}", ha="left", va="center", color="#444444", fontsize=9)

ax.set_xlim(0, max(totals) + 7)
ax.set_xlabel("Number of Labeled Studies (N = 58)", fontsize=12, fontweight="bold", labelpad=10)
ax.set_ylabel("Target Pathology", fontsize=12, fontweight="bold", labelpad=10)
ax.set_title("Target Class Distribution: Positive vs Negative Ratios", fontsize=14, fontweight="bold", pad=25)

# Place legend above the plot to avoid overlapping bars
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True, facecolor="white", framealpha=0.95, fontsize=10)

plt.tight_layout()
plt.show()


### Co-occurrence Matrix (Pearson Correlation)

In [ ]:
# Display correlation across target pathologies
labeled_df[TARGET_COLUMNS].corr().round(2)

In [ ]:
# Create Heatmap graph to check the data more intuitively
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Apply ggplot style
plt.style.use("ggplot")

# Compute Pearson correlation co-occurrence matrix
corr = labeled_df[TARGET_COLUMNS].corr()

# Define two-color diverging colormap (blue: negative, red: positive)
cmap = LinearSegmentedColormap.from_list("blue_red", ["#2166ac", "#f7f7f7", "#b2182b"])

# Plot co-occurrence heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1.0,
    vmax=1.0,
    center=0.0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation", "shrink": 0.8},
)
plt.title("Target Pathology Co-occurrence Heatmap", fontsize=14, pad=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. MRI Series & Sequence Analysis
Analyzing anatomical planes (Sagittal, Coronal, Axial) and series counts per study.

In [ ]:
# Anatomical plane breakdown
print("Anatomical Plane Distribution:")
print(series_df["Anatomical_Plane"].value_counts(normalize=True) * 100)

# Series per study stats
series_per_study = series_df.groupby("StudyInstanceUID").size()
print("\nSeries per Study Description:")
print(series_per_study.describe())

# Verify Fluid_Sensitive vs Fat_Suppression
print("\nAre Fluid_Sensitive and Fat_Suppression identical?", (series_df["Fluid_Sensitive"] == series_df["Fat_Suppression"]).all())

![Axial Coronal Sagittal Planes](https://o.quizlet.com/.m6Vj.C-ciyE-X-Q8lmAWA.png)

### Axial Plane
![Axial Plane](../data/sample/jpg/study-c70aa8373f0ff3e8.jpg)

### Sagittal Plane
![Sagittal Plane](../data/sample/jpg/study-b3753a5509cd30a8.jpg)

### Coronal Plane
![Coronal Plane](../data/sample/jpg/study-b42e5ffecc3572bb.jpg)

> Not sure about this one, tho!

## 4. Multilingual Radiology Reports
Inspecting report lengths and language distributions.

In [ ]:
train_df["char_count"] = train_df["Report"].astype(str).str.len()
train_df["word_count"] = train_df["Report"].astype(str).apply(lambda x: len(x.split()))
print(train_df[["char_count", "word_count"]].describe())

# Sample preview
print("\nSample Report:\n", train_df.iloc[0]["Report"][:400])

In [ ]:
# print(train_df[train_df["Report"].str.split(" ").str.len() < 20]["Report"].iloc[0])
# print(train_df[train_df["Report"].str.split(" ").str.len() < 20]["Report"].iloc[1])
# print(train_df[train_df["Report"].str.split(" ").str.len() < 20]["Report"].iloc[2])
# print(train_df[train_df["Report"].str.split(" ").str.len() < 20]["Report"].iloc[3])
# print(train_df[train_df["Report"].str.split(" ").str.len() < 20]["Report"].iloc[3])
# print(train_df[train_df["Report"].str.split(" ").str.len() > 600]["Report"].iloc[0])
print(train_df[train_df["Report"].str.split(" ").str.len() == 129]["Report"].iloc[0])

In [ ]:
# ------------------------------------------------------------------------------
# Multilingual Report Language Detection & Summarization
# ------------------------------------------------------------------------------
try:
    import langdetect
    from langdetect import DetectorFactory
    DetectorFactory.seed = 0
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langdetect"])
    import langdetect
    from langdetect import DetectorFactory
    DetectorFactory.seed = 0

# ISO 639-1 language code to readable name mapping
LANG_MAP = {
    "en": "English",
    "es": "Spanish",
    "tr": "Turkish",
    "hr": "Croatian",
    "el": "Greek",
    "de": "German",
    "bg": "Bulgarian",
    "nl": "Dutch",
    "fr": "French",
}

def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return np.nan
    try:
        lang = langdetect.detect(text)
        return lang if lang in LANG_MAP else np.nan
    except Exception:
        return np.nan

# Detect language for each report in train.csv
train_df["language_code"] = train_df["Report"].apply(detect_language)
train_df["language"] = train_df["language_code"].map(LANG_MAP).fillna("nan")

# Summarize report counts per language
lang_counts = train_df["language"].value_counts(dropna=False).reset_index()
lang_counts.columns = ["Language", "Report Count"]
lang_counts["Percentage (%)"] = ((lang_counts["Report Count"] / len(train_df)) * 100).round(2)

print("=== Number of Reports by Language ===")
print(lang_counts.to_string(index=False))


In [ ]:
# Visualize language distribution
import matplotlib.pyplot as plt

plt.style.use("ggplot")

fig, ax = plt.subplots(figsize=(10, 6))

df_plot = lang_counts.iloc[::-1].reset_index(drop=True)
bars = ax.barh(df_plot["Language"], df_plot["Report Count"], color="#348ABD", height=0.65)

# Annotate bars with counts and percentages
for i, (cnt, pct) in enumerate(zip(df_plot["Report Count"], df_plot["Percentage (%)"])):
    ax.text(cnt + 15, i, f"{cnt:,} ({pct:.2f}%)", va="center", ha="left", color="#444444", fontsize=10, fontweight="bold")

ax.set_xlim(0, max(df_plot["Report Count"]) * 1.18)
ax.set_xlabel("Number of Reports", fontsize=12, fontweight="bold", labelpad=10)
ax.set_ylabel("Language", fontsize=12, fontweight="bold", labelpad=10)
ax.set_title(f"Multilingual Radiology Reports Distribution (N = {len(train_df):,})", fontsize=14, fontweight="bold", pad=15)

plt.tight_layout()
plt.show()


## 5. Sample DICOM Inspection

In [ ]:
for f in glob.glob("data/sample/dcm/*.dcm"):
    ds = pydicom.dcmread(f)
    print(f"File: {os.path.basename(f)}")
    print(f"  Modality:        {ds.get('Modality')}")
    print(f"  Matrix:          {ds.Rows}x{ds.Columns}")
    print(f"  Pixel Spacing:   {ds.get('PixelSpacing')}")
    print(f"  Slice Thickness: {ds.get('SliceThickness')} mm\n")

## Actual Data Folder Structure
```
mri_project/
└── raw_dicom/
    ├── patient_000001/
    │   ├── study_001/
    │   │   ├── series_001/
    │   │   │   ├── image_0001.dcm
    │   │   │   ├── image_0002.dcm
    │   │   │   └── ...
    │   │   └── series_002/
    │   │       └── ...
    │   └── study_002/
    │       └── ...
    └── patient_000002/
        └── ...
```

* Patient: The same patient labeled with ID
* Study: 

## Batch convert niix to dcm
This saves about 1.5x in size generally.
```
dcm2niix \
  -z y \
  -f "%p_%s" \
  -o data/sample/nii \
  data/sample/dcm/
```

In [ ]:
labeled_df.to_csv(DATA_DIR / "cleaned.csv")
labeled_df[TARGET_COLUMNS] = np.nan
labeled_df.to_csv(DATA_DIR / "drop.csv")

## Prompt to Generate Medical Label
```
Based on the medical report in the `drop.csv` file, label the degree of symdrome for each class and case (from 0 to 1, shows the likelihood of actually have such symdrome or disease). Create a new file called `lm_label.csv`. Do not modify the column name and row name.
```

## 6. LLM Label Quality vs Gold

Compare soft pathology probabilities in `data/lm/*.csv` against binary gold in `data/cleaned.csv`.

- **Join key:** `StudyInstanceUID`
- **Soft metrics:** AUROC, AUPRC, **MAE** (LM as probabilities; MAE is the soft error metric)
- **Hard snapshot:** F1 at threshold 0.5
- **Layout:** overall summary (one row per model), then per-pathology tables (models as columns)


In [ ]:
from IPython.display import display

GOLD_PATH = DATA_DIR / "cleaned.csv"
LM_DIR = DATA_DIR / "lm_label"
JOIN_KEY = "StudyInstanceUID"
HARD_THRESHOLD = 0.5


def _safe_auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """AUROC; NaN if gold has only one class."""
    y_true = y_true.astype(float)
    if np.unique(y_true).size < 2:
        return np.nan
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    sum_pos_ranks = ranks[y_true == 1].sum()
    return float((sum_pos_ranks - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def _safe_auprc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Average precision (AUPRC); NaN if no positives."""
    y_true = y_true.astype(float)
    n_pos = y_true.sum()
    if n_pos == 0:
        return np.nan
    order = np.argsort(-y_score)
    y_true_sorted = y_true[order]
    tp = np.cumsum(y_true_sorted)
    fp = np.cumsum(1 - y_true_sorted)
    precision = tp / (tp + fp)
    # AP over positive ranks only
    return float((precision * y_true_sorted).sum() / n_pos)


def _f1_at_threshold(y_true: np.ndarray, y_score: np.ndarray, thr: float = HARD_THRESHOLD) -> float:
    y_pred = (y_score >= thr).astype(float)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    if tp + fp == 0 or tp + fn == 0:
        return np.nan if tp == 0 else 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if precision + recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


def score_lm_vs_gold(gold: pd.DataFrame, pred: pd.DataFrame, model_id: str) -> pd.DataFrame:
    """Per-pathology metrics for one LM file, aligned on StudyInstanceUID."""
    merged = gold[[JOIN_KEY] + TARGET_COLUMNS].merge(
        pred[[JOIN_KEY] + TARGET_COLUMNS],
        on=JOIN_KEY,
        suffixes=("_gold", "_lm"),
        how="inner",
        validate="one_to_one",
    )
    rows = []
    for col in TARGET_COLUMNS:
        y_true = merged[f"{col}_gold"].to_numpy(dtype=float)
        y_score = merged[f"{col}_lm"].to_numpy(dtype=float)
        rows.append(
            {
                "pathology": col,
                "model": model_id,
                "n": len(merged),
                "auroc": _safe_auroc(y_true, y_score),
                "auprc": _safe_auprc(y_true, y_score),
                "mae": float(np.mean(np.abs(y_score - y_true))),
                "f1@0.5": _f1_at_threshold(y_true, y_score),
            }
        )
    return pd.DataFrame(rows)


gold_df = pd.read_csv(GOLD_PATH)
lm_paths = sorted(LM_DIR.glob("*.csv"))
if not lm_paths:
    raise FileNotFoundError(f"No LM CSVs found under {LM_DIR}")

per_pathology = pd.concat(
    [
        score_lm_vs_gold(gold_df, pd.read_csv(path), model_id=path.stem)
        for path in lm_paths
    ],
    ignore_index=True,
)

# --- Overall: one row per model (macro means; NaNs skipped) ---
overall = (
    per_pathology.groupby("model", sort=True)
    .agg(
        n_studies=("n", "first"),
        macro_auroc=("auroc", "mean"),
        macro_auprc=("auprc", "mean"),
        mean_mae=("mae", "mean"),
        macro_f1_at_0_5=("f1@0.5", "mean"),
    )
    .round(3)
)

print("Overall (macro over pathologies; NaN pathologies excluded from each mean)")
display(overall)

# --- Per pathology: models as columns ---
def pivot_metric(metric: str) -> pd.DataFrame:
    table = (
        per_pathology.pivot(index="pathology", columns="model", values=metric)
        .reindex(TARGET_COLUMNS)
        .round(3)
    )
    table.index.name = "pathology"
    return table


print("\nPer-pathology AUROC (higher better; NaN = only one gold class)")
display(pivot_metric("auroc"))

print("\nPer-pathology AUPRC (higher better; NaN = no gold positives)")
display(pivot_metric("auprc"))

print("\nPer-pathology MAE (lower better; soft probability vs binary gold)")
display(pivot_metric("mae"))

print("\nPer-pathology F1 @ 0.5 (higher better; NaN if undefined)")
display(pivot_metric("f1@0.5"))

# Sanity: row counts / join coverage
coverage = pd.DataFrame(
    [
        {
            "model": path.stem,
            "lm_rows": len(pd.read_csv(path)),
            "gold_rows": len(gold_df),
            "joined_rows": int(per_pathology.loc[per_pathology["model"] == path.stem, "n"].iloc[0]),
        }
        for path in lm_paths
    ]
)
print("\nJoin coverage")
display(coverage)


### Notes

> Gemini 3.1 pro high sent the work to 3 subagents to run, I didn't confirm the exact model used by agents.
